# AlphaFold2 Structure Prediction via ColabFold

This notebook predicts structures for SA-generated and stored protein sequences
using AlphaFold2 (via ColabFold). Upload the `colabfold_input/` directory before running.

**Runtime:** Select GPU runtime (Runtime → Change runtime type → T4 GPU)

In [ ]:
# Cell 1: Install ColabFold (from official ColabFold notebook)
import os
if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
    # Delete the broken TensorFlow .so file (official ColabFold fix)
    os.system("rm -f /usr/local/lib/python3.*/dist-packages/tensorflow/core/kernels/libtfkernel_sobol_op.so")
    os.system("touch COLABFOLD_READY")
    print("Done! If you see dependency warnings above, ignore them.")
else:
    print("ColabFold already installed.")

In [ ]:
# Cell 1b: Mount Google Drive for automatic backups
from google.colab import drive
drive.mount('/content/drive')

BACKUP_DIR = "/content/drive/MyDrive/colabfold_backup"
import os
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"Backup directory: {BACKUP_DIR}")

In [ ]:
# Cell 3: Run predictions using ColabFold Python API
import glob, os, shutil
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

setup_logging(Path("log.txt"))

# Download AlphaFold2 model weights (cached after first download)
download_alphafold_params("alphafold2_ptm", Path(default_data_dir))

BACKUP_DIR = "/content/drive/MyDrive/colabfold_backup"

def backup_family(fam_id, cat):
    """Copy completed results to Google Drive."""
    src = f"colabfold_output/{fam_id}/{cat}"
    dst = f"{BACKUP_DIR}/{fam_id}/{cat}"
    os.makedirs(dst, exist_ok=True)
    n_copied = 0
    for f in glob.glob(f"{src}/*.pdb"):
        shutil.copy2(f, dst)
        n_copied += 1
    print(f"  Backed up {n_copied} PDB files to Drive: {fam_id}/{cat}")

FAMILIES = [
    "PF00397",   # WW (~23 residues)
    "PF00096",   # zf-C2H2 (~23 residues)
    "PF00018",   # SH3 (~60 residues)
    "PF00014",   # Kunitz (~58 residues)
    "PF00076",   # RRM (~70 residues)
    "PF00595",   # PDZ (~90 residues)
    "PF00069",   # Pkinase (~262 residues)
]
CATEGORIES = ["sa_generation", "stored"]

for fam_id in FAMILIES:
    for cat in CATEGORIES:
        fasta = f"colabfold_input/{fam_id}/{cat}.fasta"
        outdir = f"colabfold_output/{fam_id}/{cat}"

        if not os.path.isfile(fasta):
            print(f"  {fam_id}/{cat}: FASTA not found, skipping")
            continue

        n_input = sum(1 for l in open(fasta) if l.startswith('>'))
        os.makedirs(outdir, exist_ok=True)
        n_done = len(glob.glob(f"{outdir}/*rank_001*.pdb"))

        if n_done >= n_input:
            print(f"  {fam_id}/{cat}: DONE ({n_done}/{n_input})")
            continue

        # Check if backup has results we can restore
        backup_pdbs = glob.glob(f"{BACKUP_DIR}/{fam_id}/{cat}/*.pdb")
        if backup_pdbs and n_done == 0:
            print(f"  Restoring {len(backup_pdbs)} PDB files from Drive backup...")
            for f in backup_pdbs:
                shutil.copy2(f, outdir)
            n_done = len(glob.glob(f"{outdir}/*rank_001*.pdb"))
            if n_done >= n_input:
                print(f"  {fam_id}/{cat}: RESTORED ({n_done}/{n_input})")
                continue

        print(f"\n{'='*60}")
        print(f"  {fam_id}/{cat}: {n_input} sequences ({n_done} done)")
        print(f"{'='*60}")

        queries, is_complex = get_queries(fasta)
        run(
            queries=queries,
            result_dir=outdir,
            use_templates=False,
            num_relax=0,
            msa_mode="MMseqs2 (UniRef+Environmental)",
            model_type="alphafold2_ptm",
            num_models=1,
            num_recycles=3,
            model_order=[1],
            is_complex=is_complex,
            data_dir=Path(default_data_dir),
            keep_existing_results=True,
            rank_by="auto",
            stop_at_score=float(100),
            zip_results=False,
            user_agent="colabfold/google-colab-main",
        )

        # Backup to Drive after each family/category completes
        backup_family(fam_id, cat)

print("\nAll predictions complete!")

In [ ]:
# Cell 3: Run predictions using ColabFold Python API
import glob, os
from pathlib import Path
from colabfold.download import download_alphafold_params, default_data_dir
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

setup_logging(Path("log.txt"))

# Download AlphaFold2 model weights (cached after first download)
download_alphafold_params("alphafold2_ptm", Path(default_data_dir))

FAMILIES = [
    "PF00397",   # WW (~23 residues)
    "PF00096",   # zf-C2H2 (~23 residues)
    "PF00018",   # SH3 (~60 residues)
    "PF00014",   # Kunitz (~58 residues)
    "PF00076",   # RRM (~70 residues)
    "PF00595",   # PDZ (~90 residues)
    "PF00069",   # Pkinase (~262 residues)
]
CATEGORIES = ["sa_generation", "stored"]

for fam_id in FAMILIES:
    for cat in CATEGORIES:
        fasta = f"colabfold_input/{fam_id}/{cat}.fasta"
        outdir = f"colabfold_output/{fam_id}/{cat}"

        if not os.path.isfile(fasta):
            print(f"  {fam_id}/{cat}: FASTA not found, skipping")
            continue

        n_input = sum(1 for l in open(fasta) if l.startswith('>'))
        os.makedirs(outdir, exist_ok=True)
        n_done = len(glob.glob(f"{outdir}/*rank_001*.pdb"))

        if n_done >= n_input:
            print(f"  {fam_id}/{cat}: DONE ({n_done}/{n_input})")
            continue

        print(f"\n{'='*60}")
        print(f"  {fam_id}/{cat}: {n_input} sequences ({n_done} done)")
        print(f"{'='*60}")

        queries, is_complex = get_queries(fasta)
        run(
            queries=queries,
            result_dir=outdir,
            use_templates=False,
            num_relax=0,
            msa_mode="MMseqs2 (UniRef+Environmental)",
            model_type="alphafold2_ptm",
            num_models=1,
            num_recycles=3,
            model_order=[1],
            is_complex=is_complex,
            data_dir=Path(default_data_dir),
            keep_existing_results=True,
            rank_by="auto",
            stop_at_score=float(100),
            zip_results=False,
            user_agent="colabfold/google-colab-main",
        )

print("\nAll predictions complete!")

In [ ]:
# Cell 4: Check completion status
import glob, os

FAMILIES = ["PF00397","PF00096","PF00018","PF00014","PF00076","PF00595","PF00069"]
CATEGORIES = ["sa_generation", "stored"]

total = 0
done = 0
for fam_id in FAMILIES:
    for cat in CATEGORIES:
        fasta = f"colabfold_input/{fam_id}/{cat}.fasta"
        outdir = f"colabfold_output/{fam_id}/{cat}"
        if os.path.isfile(fasta):
            n_in = sum(1 for l in open(fasta) if l.startswith('>'))
            n_out = len(glob.glob(f"{outdir}/*rank_001*.pdb"))
            total += n_in
            done += n_out
            status = 'DONE' if n_out >= n_in else f'{n_out}/{n_in}'
            print(f"  {fam_id}/{cat}: {status}")

print(f"\nTotal: {done}/{total} complete")

In [ ]:
# Cell 5: Download results
import shutil
from google.colab import files

shutil.make_archive('colabfold_output', 'zip', '.', 'colabfold_output')
files.download('colabfold_output.zip')
print("Download colabfold_output.zip and unzip in code/structure-validation/")